# TraceArena Investment Agent Benchmark v1

Reproduce the published deterministic contract baseline without an API key. The run uses a synthetic fixture, disables network-backed research and brokerage execution, and never places real orders. It is not financial advice.

**Claim boundary:** both bundled entrants are scripted controls. This notebook verifies the evaluation contract; it does not claim to compare models.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

RELEASE = "v0.1.12"
REPO = Path("/content/TraceArena")
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", RELEASE,
        "https://github.com/tonyhyworld/TraceArena.git", str(REPO),
    ], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Ready: TraceArena {RELEASE}")

In [ ]:
OUTPUT = Path("/content/tracearena-investment-benchmark-v1")
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "backend")
subprocess.run([
    sys.executable, "backend/scripts/investment_benchmark.py",
    "--output", str(OUTPUT),
], check=True, env=env)

In [ ]:
import json
from IPython.display import Markdown, display

generated = json.loads((OUTPUT / "benchmark_report.json").read_text())
published = json.loads((REPO / "benchmarks/investment-agent-v1/benchmark_report.json").read_text())
assert generated == published, "Generated report differs from the published v0.1.12 baseline"
assert generated["official_model_leaderboard"] is False
assert generated["execution_boundary"]["network"] == "disabled"
assert generated["execution_boundary"]["brokerage"] == "disabled"
display(Markdown((OUTPUT / "LEADERBOARD.md").read_text()))
print("Verified report SHA-256:", generated["report_sha256"])

## What to share

Please post your Colab runtime, Python version, and verified report SHA-256 in [Discussion #45](https://github.com/tonyhyworld/TraceArena/discussions/45). A failed reproduction is equally useful.